In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
def softmax(X):
  norm = np.sum(np.exp(X))
  Y = np.exp(X)/norm
  return Y

def normalise(X):
  X= X/np.sum(X,0)
  return X

In [ ]:
8#@title
#s1con = np.zeros(4) #context (1,2,3,4)
s1loc = np.zeros((4,4)) #location (1,2,3,4) x time (1,2,3,4)
#o1cue = np.zeros(4) #cues
o2rew = np.zeros(4) #reward

D2aff = np.array([0.5,0.5]) #prior of valence (pos/neg)
D2exh = np.array([0.2,.8]) #prior of exhaustion (high/low)
D2hun = np.array([0.9,0.1]) #prior of hunger (high/low)

betam = np.array([0.5,2])

#D2con = np.array([0.09,0.10,0.90,0.01])

s2aff = D2aff # valence (pos/neg)
#s2con = D2con #context (1,2,3,4)
s2exh = D2exh #exhaustion (++,+,-,--)??
s2hun = D2hun #hunger (++,+,-,--)

A1 = np.zeros((4,4)) #likelihood of outcome (1,2,3,4) given loc (1,2,3,4)

#A2exh = np.zeros((2,4)) #likelihood of outcome s2aff (pos/neg), given s2hun (4)
#A2hun = np.zeros((2,4)) #likelihood of outcome s2aff (pos/neg), given s2exh (4)

B1 = np.zeros((4,4,4)) #state transitions (1,2,3,4)
D1 = np.array([1,0,0,0])
for i in range(4):
  #A1[i,i] = 1
  for j in range(4):

    if i ==j:
      B1[i,:,j] = 0.8
      A1[i,j] = 1#0.8
    if abs(i-j)==1:
      B1[i,:,j] = 0.2
      A1[i,j] = 0#0.015
    if abs(i-j)==2:
      B1[i,:,j] = 0.02
      A1[i,j] = 0#.01
    if abs(i-j)==3:
      B1[i,:,j] = 0.005
      A1[i,j] = 0#.0005

B1[3,3,:]=1.0
B1[0:3,3,:]=0

B2aff = np.zeros((2,2)) #likelihood to transition to other states (potentially even hun and loc?)
B2aff[:,0]=[0.99,0.01] ##probability of feeling good later (yes/no) given good now
B2aff[:,1]=[0.02,0.98] ##probability of feeling good later (yes/no) given bad now


B2exh = np.zeros((2,2)) #likelihood to transition to other states (potentially even hun and loc?)
B2exh[:,0]=[0.95,0.05] ##probability of feeling exhausted later (yes/no) given exhausted now
B2exh[:,1]=[0,1] ##probability of feeling exhausted later (yes/no) given not exhausted now

B2hun = np.zeros((2,2))
B2hun[:,0]=[1.0,0] ##probability of feeling hungry later (yes/no) given hungry now
B2hun[:,1]=[0.05,0.95] ##probability of feeling hungry later (yes/no) given not hungry now


for i in range(4):
  B1[:,:,i] = normalise(B1[:,:,i])#0**gammaB[0])
A1 = normalise(A1)

C0 = np.array([0,1,2,-3]) ### rewards
C1energy = np.array([2,-1,-3,-3]) ## energy preferences
C1hunger = np.array([-2,4,8,-2]) ## food preferences

C1 = np.reshape(C0 + s2exh[0]*C1energy + s2hun[0]*C1hunger,4) #< Casper
#maybe np.dot? np.dot(np.array([[2,0]]),np.array([2,0]))

#C2urg = #softmax*preference comparison: s2exh[0,0]*C1energy - s2hun[0,0]*C1hunger
gammat = np.zeros(3) #action model precision for each fork

global II
II = 0

#def forward (s,o,G):
#  II+= 1
#
#  if II == 4:
##    II=0
#    return G
##  else:
 #   Gu = np.zeros(4)
#    for j in range(4): ##loop over actions
#      s = np.inner(s,B1[:,:,j])
#      o = np.inner(s,A1)
#      Gu[j] += np.dot(o2u1[:,j],(np.log(o2u1[:,j])+C1)) ##first action

#      G[j] 

In [ ]:
N = 2000 #number of samples
M = 100 #number of sampled agents
gamma_group = np.zeros((M,N+1))
uo1T_group = np.zeros((M,4,N))
s2aff_group = np.zeros((M,2,N+1))
s2exh_group = np.zeros((M,2,N+1))
s2hun_group = np.zeros((M,2,N+1))

for l in range(M):
  s2aff = np.zeros((2,N+1))
  s2aff[:,0] = D2aff

  s2exh = np.zeros((2,N+1))
  s2exh[:,0] = D2exh

  s2hun = np.zeros((2,N+1))
  s2hun[:,0] = D2hun

  #s2urg = np.zeros((2,N+1))
  #s2urg[:,0] = D2urg

  t1 = np.zeros(4) #o1
  t2 = np.zeros((4,4,4)) #o1,u1,o2
  t3 = np.zeros((4,4,4,4,4)) #o1,u1,o2,u2,o3

  G1 = np.zeros((4,4)) ##G given outcome1,action1
  G2 = np.zeros((4,4,4,4)) ##G given past + outcome2,action2
  G3 = np.zeros((4,4,4,4,4,4)) ##G given past + outcome3,action3

  seq = np.zeros((7,N)) ##store sequences, *change 7?

  uo1T = np.zeros((4,N))
  uo2T = np.zeros((4,N))
  uo3T = np.zeros((4,N))

  ou2T = np.zeros((4,N))
  ou3T = np.zeros((4,N))
  ou4T = np.zeros((4,N))


  for i in range(N): ##samples
    C1 = np.reshape(C0 + s2exh[0,i]*C1energy + s2hun[0,i]*C1hunger,4)
    #print("sample:",i,C1)
    Eaff = np.zeros(2)
    Eexh = np.zeros(2)
    Ehun = np.zeros(2)
  
    beta = np.dot(s2aff[:,i],betam)
    gammat[:] = beta**-1

    eAC = np.zeros(3)

    ##t = 1
    o1 = np.random.choice(range(4),p=np.inner(D1,A1)) #*

    Eexh[1]+= C1energy[o1] 
    Ehun[1]+= C1hunger[o1]
    Eexh[0]+= -1*C1energy[o1] 
    Ehun[0]+= -1*C1hunger[o1]

    s1 = softmax(np.log(D1+10**-7) + np.log(A1[o1,:]+10**-7))

    #s2u1 = np.zeros((4,4)) ##states2 given action at time 1
    #o2u1 = np.zeros((4,4)) ##outcomes2 given action at time 1

    if t1[o1] == 0:
      for j in range(4):
        s2u1 = np.inner(s1,B1[:,:,j])
        o2u1 = np.inner(s2u1,A1)

        G1[o1,j] += np.dot(o2u1,(np.log(o2u1+10**-7)+C1))
      t1[o1]=1 ##mark path
    uo1 = softmax(gammat[0]*G1[o1,:])
    u1 = np.random.choice(range(4),p=uo1)



    uo1T[:,i] = uo1
    uo1[u1] +=-1
    eAC[0] = np.dot(uo1,G1[o1,:])


    ###t=2
    s2 = np.inner(s1,B1[:,:,u1])
    ou2 = np.inner(s2,A1)
    ou2T[:,i]=ou2

    o2 = np.random.choice(range(4),p=ou2)

    Eexh[1]+= C1energy[o2] 
    Ehun[1]+= C1hunger[o2]

    Eexh[0]+= -1*C1energy[o2] 
    Ehun[0]+= -1*C1hunger[o2]

    s2 = softmax(np.log(s2+10**-7) + np.log(A1[o2,:]+10**-7))

    #G2 = np.zeros(4) ##G given action1
    if t2[o1,u1,o2]==0:
      for j in range(4):
        s3u2 = np.inner(s2,B1[:,:,j])
        o3u2 = np.inner(s3u2,A1)
        #G2[o1,u1,o2,j] += np.dot(o3u2,(np.log(o3u2+10**-7)+C1))
        G2[o1,u1,o2,j] += np.dot(o3u2,(np.log(o3u2+10**-7)))
  
    uo2 = softmax(gammat[1]*G2[o1,u1,o2,:])
    uo2T[:,i] = uo2
    if t2[o1,u1,o2]==0:
      G1[o1,u1] += ou2[o2]*np.dot(uo2,G2[o1,u1,o2,:])
      t2[o1,u1,o2]=1 
    u2 = np.random.choice(range(4),p=uo2) ##sample action2


    uo2[u2] +=-1
    eAC[1] = np.dot(uo2,G2[o1,u1,o2,:])
  
    ###t=3
    s3 = np.inner(s2,B1[:,:,u2])
    ou3=np.inner(s3,A1)
    ou3T[:,i]=ou3
    o3 = np.random.choice(range(4),p=ou3) ###sample outcome3

    Eexh[1]+= C1energy[o3] 
    Ehun[1]+= C1hunger[o3]

    Eexh[0]+= -1*C1energy[o3] 
    Ehun[0]+= -1*C1hunger[o3]

    s3 = softmax(np.log(s3+10**-7) + np.log(A1[o3,:]+10**-7))
    #G3 = np.zeros(4) ##G given action1

    if t3[o1,u1,o2,u2,o3]==0:
      for j in range(4):
        s4u3 = np.inner(s3,B1[:,:,j])
        o4u3 = np.inner(s4u3,A1)
        G3[o1,u1,o2,u2,o3,j] += np.dot(o4u3,(np.log(o4u3+10**-7)+C1))
      #G1[o1,:] += G3[o1,u1,o2,u2,o3,:]
      #G2[o1,u1,o2,:] += G3[o1,u1,o2,u2,o3,:]
    
    uo3 = softmax(gammat[2]*G3[o1,u1,o2,u2,o3,:])
    uo3T[:,i] = uo3
    if t3[o1,u1,o2,u2,o3]==0:
      Gadd = ou3[o3]*np.dot(uo3,G3[o1,u1,o2,u2,o3,:])
      G1[o1,u1] += ou2[o2]*Gadd
      G2[o1,u1,o2,u2] += Gadd
      t3[o1,u1,o2,u2,o3]=1

    u3 = np.random.choice(range(4),p=uo3) ##sample action3
    uo3[u3] +=-1 ##calc diff prior post
    eAC[2] = np.dot(uo3,G3[o1,u1,o2,u2,o3,:])
  

    ###t=4
    s4 = np.inner(s3,B1[:,:,u3])

    ou4 = np.inner(s4,A1)
    ou4T[:,i]=ou4
    o4 = np.random.choice(range(4),p=ou4) ###sample outcome4
    s4 = softmax(np.log(s4+10**-7) + np.log(A1[o4,:]+0.01**-7))

    Eexh[1]+= C1energy[o4] 
    Ehun[1]+= C1hunger[o4]

    Eexh[0]+= -1*C1energy[o4] 
    Ehun[0]+= -1*C1hunger[o4]

    for t in range(3):
      Eaff += -1*np.log((betam-eAC[t])/betam+10**-7)+np.log(1-eAC[t]*beta**-1+10**-7)

    Eaff[np.isnan(Eaff)]=0

    ####effect of evidence from present trial:
    s2aff[:,i] = softmax(np.log(s2aff[:,i]+10**-7) + 0.2*Eaff)
    s2exh[:,i] = softmax(np.log(s2exh[:,i]+10**-7) + 0.1*Eexh)
    s2hun[:,i] = softmax(np.log(s2hun[:,i]+10**-7) + 0.1*Ehun)

    ####effect of evidence from present trial:
    s2aff[:,i+1] = np.inner(s2aff[:,i],B2aff)
    s2exh[:,i+1] = np.inner(s2exh[:,i],B2exh)
    s2hun[:,i+1] = np.inner(s2hun[:,i],B2hun)

    seq[:,i] = [o1,u1,o2,u2,o3,u3,o4]

  gamma_group[l,:] = s2aff[0,:]*betam[0]**-1 + s2aff[1,:]*betam[1]**-1
  uo1T_group[l,:,:] = uo1T
  s2aff_group[l,:,:] = s2aff
  s2exh_group[l,:,:] = s2exh
  s2hun_group[l,:,:] = s2hun


/usr/local/lib/python3.7/dist-packages/ipykernel_launcher.py:168: RuntimeWarning: invalid value encountered in log


In [ ]:
group_cutoff = round(M*0.15)
group_mid_cutoff = M - group_cutoff * 2
gamma_group_sums = np.array(gamma_group.sum(axis=1))
gamma_group_sums_ordered_ind = np.argsort(np.argsort(gamma_group_sums))

bool_low_gamma = gamma_group_sums_ordered_ind <= group_cutoff - 1
bool_mid_gamma = np.where((gamma_group_sums_ordered_ind > group_cutoff - 1) &
                          (gamma_group_sums_ordered_ind < M - group_cutoff),
                          True, False)  
bool_high_gamma = gamma_group_sums_ordered_ind >= M - group_cutoff

gamma_group_low = gamma_group[bool_low_gamma,:].sum(axis=0)/group_cutoff
gamma_group_mid = gamma_group[bool_mid_gamma,:].sum(axis=0)/group_mid_cutoff
gamma_group_high = gamma_group[bool_high_gamma,:].sum(axis=0)/group_cutoff

avg_std_gamma_low = gamma_group[bool_low_gamma,:].std(axis=1).mean(axis=0)
avg_std_gamma_mid = gamma_group[bool_mid_gamma,:].std(axis=1).mean(axis=0)
avg_std_gamma_high = gamma_group[bool_high_gamma,:].std(axis=1).mean(axis=0)

#mean & std

SummaryGammaGroup = pd.DataFrame({'Low':[np.mean(gamma_group_low),
                                         avg_std_gamma_low],
                                  'Medium':[np.mean(gamma_group_mid),
                                            avg_std_gamma_mid],
                                  'High':[np.mean(gamma_group_high),
                                          avg_std_gamma_high]},
                                 index=['Mean', 'Avg.Std'])
print(SummaryGammaGroup)

#plt.figure(figsize=(8,16))
#plt.subplot(3,1,1)
plt.plot(gamma_group_low)#, label='$\gamma$')
plt.title('action model precision($\gamma$)')
plt.xticks([0,500,1000,1500,2000],[])
plt.plot(gamma_group_mid)#, label='$\gamma$')
plt.title('action model precision($\gamma$)')
plt.xticks([0,500,1000,1500,2000],[])
plt.plot(gamma_group_high)#, label='$\gamma$')
plt.title('action model precision($\gamma$)')
plt.xticks([0,500,1000,1500,2000],[])

In [ ]:
uo1T_group_low = np.zeros((4,N))
uo1T_group_mid = np.zeros((4,N))
uo1T_group_high = np.zeros((4,N))
avg_std_uo1T_group = np.zeros((4,3))
for i in range(4):
  uo1T_group_low[i] = uo1T_group[bool_low_gamma,i,:].mean(axis=0)
  uo1T_group_mid[i] = uo1T_group[bool_mid_gamma,i,:].mean(axis=0)
  uo1T_group_high[i] = uo1T_group[bool_high_gamma,i,:].mean(axis=0)
  avg_std_uo1T_group[i,0] = uo1T_group[bool_low_gamma,i,:].std(axis=1).mean(axis=0)
  avg_std_uo1T_group[i,1] = uo1T_group[bool_mid_gamma,i,:].std(axis=1).mean(axis=0)
  avg_std_uo1T_group[i,2] = uo1T_group[bool_high_gamma,i,:].std(axis=1).mean(axis=0)

#mean & std

SummaryUo1TC0 = pd.DataFrame({'Low':[uo1T_group_low[0].mean(axis=0),avg_std_uo1T_group[0,0]],
                              'Mid':[uo1T_group_mid[0].mean(axis=0),avg_std_uo1T_group[0,1]],
                              'High':[uo1T_group_high[0].mean(axis=0),avg_std_uo1T_group[0,2]]},
                             index=['Mean', 'Avg.Std'])

SummaryUo1TC1 = pd.DataFrame({'Low':[uo1T_group_low[1].mean(axis=0),avg_std_uo1T_group[1,0]],
                              'Mid':[uo1T_group_mid[1].mean(axis=0),avg_std_uo1T_group[1,1]],
                              'High':[uo1T_group_high[1].mean(axis=0),avg_std_uo1T_group[1,2]]},
                             index=['Mean', 'Avg.Std'])

SummaryUo1TC2 = pd.DataFrame({'Low':[uo1T_group_low[2].mean(axis=0),avg_std_uo1T_group[2,0]],
                              'Mid':[uo1T_group_mid[2].mean(axis=0),avg_std_uo1T_group[2,1]],
                              'High':[uo1T_group_high[2].mean(axis=0),avg_std_uo1T_group[2,2]]},
                             index=['Mean', 'Avg.Std'])

SummaryUo1TCneg1 = pd.DataFrame({'Low':[uo1T_group_low[3].mean(axis=0),avg_std_uo1T_group[3,0]],
                                 'Mid':[uo1T_group_mid[3].mean(axis=0),avg_std_uo1T_group[3,1]],
                                 'High':[uo1T_group_high[3].mean(axis=0),avg_std_uo1T_group[3,2]]},                                                                 
                                 index=['Mean', 'Avg.Std'])
print(SummaryUo1TC0)
print(SummaryUo1TC1)
print(SummaryUo1TC2)
print(SummaryUo1TCneg1)

labels = ['C=0','C=1', 'C=2', 'C=-2']
plt.figure(figsize=(8,16))
plt.subplot(4,1,1)
for i in range(4):
  plt.plot(np.arange(N),uo1T_group_low[i,:],label=labels[i])

plt.title("$u_{1} = \sigma(\gamma G_1)$")
plt.legend()
plt.xticks([0,500,1000,1500,2000,2500],[])
plt.subplot(4,1,2)
for i in range(4):
  plt.plot(np.arange(N),uo1T_group_mid[i,:],label=labels[i])

plt.title("$u_{1} = \sigma(\gamma G_1)$")
plt.legend()
plt.xticks([0,500,1000,1500,2000,2500],[])
plt.subplot(4,1,3)
for i in range(4):
  plt.plot(np.arange(N),uo1T_group_high[i,:],label=labels[i])

plt.title("$u_{1} = \sigma(\gamma G_1)$")
plt.legend()
plt.xticks([0,500,1000,1500,2000,2500],[])

In [ ]:

s2aff_group_low = s2aff_group[bool_low_gamma,0,:].mean(axis=0)
s2exh_group_low = s2exh_group[bool_low_gamma,0,:].mean(axis=0)
s2hun_group_low = s2hun_group[bool_low_gamma,0,:].mean(axis=0)

mean_s2aff_group_low = s2aff_group_low.mean()
mean_s2exh_group_low = s2exh_group_low.mean()
mean_s2hun_group_low = s2hun_group_low.mean()

avg_std_s2aff_group_low = s2aff_group[bool_low_gamma,0,:].std(axis=1).mean()
avg_std_s2exh_group_low = s2exh_group[bool_low_gamma,0,:].std(axis=1).mean()
avg_std_s2hun_group_low = s2hun_group[bool_low_gamma,0,:].std(axis=1).mean()

s2aff_group_mid = s2aff_group[bool_mid_gamma,0,:].mean(axis=0)
s2exh_group_mid = s2exh_group[bool_mid_gamma,0,:].mean(axis=0)
s2hun_group_mid = s2hun_group[bool_mid_gamma,0,:].mean(axis=0)

mean_s2aff_group_mid = s2aff_group_mid.mean()
mean_s2exh_group_mid = s2exh_group_mid.mean()
mean_s2hun_group_mid = s2hun_group_mid.mean()

avg_std_s2aff_group_mid = s2aff_group[bool_mid_gamma,0,:].std(axis=1).mean()
avg_std_s2exh_group_mid = s2exh_group[bool_mid_gamma,0,:].std(axis=1).mean()
avg_std_s2hun_group_mid = s2hun_group[bool_mid_gamma,0,:].std(axis=1).mean()

s2aff_group_high = s2aff_group[bool_high_gamma,0,:].mean(axis=0)
s2exh_group_high = s2exh_group[bool_high_gamma,0,:].mean(axis=0)
s2hun_group_high = s2hun_group[bool_high_gamma,0,:].mean(axis=0)

mean_s2aff_group_high = s2aff_group_high.mean()
mean_s2exh_group_high = s2exh_group_high.mean()
mean_s2hun_group_high = s2hun_group_high.mean()

avg_std_s2aff_group_high = s2aff_group[bool_high_gamma,0,:].std(axis=1).mean()
avg_std_s2exh_group_high = s2exh_group[bool_high_gamma,0,:].std(axis=1).mean()
avg_std_s2hun_group_high = s2hun_group[bool_high_gamma,0,:].std(axis=1).mean()

SummaryAffect = pd.DataFrame({'Low':[mean_s2aff_group_low,
                                     avg_std_s2aff_group_low],
                                  'Medium':[mean_s2aff_group_mid,
                                            avg_std_s2aff_group_mid],
                                  'High':[mean_s2aff_group_high,
                                          avg_std_s2aff_group_high]},
                                 index=['Mean', 'Avg.Std'])
print(SummaryAffect)

SummaryExhaustion = pd.DataFrame({'Low':[mean_s2exh_group_low,
                                     avg_std_s2exh_group_low],
                                  'Medium':[mean_s2exh_group_mid,
                                            avg_std_s2exh_group_mid],
                                  'High':[mean_s2exh_group_high,
                                          avg_std_s2exh_group_high]},
                                 index=['Mean', 'Avg.Std'])
print(SummaryExhaustion)

SummaryHunger = pd.DataFrame({'Low':[mean_s2hun_group_low,
                                     avg_std_s2hun_group_low],
                                  'Medium':[mean_s2hun_group_mid,
                                            avg_std_s2hun_group_mid],
                                  'High':[mean_s2hun_group_high,
                                          avg_std_s2hun_group_high]},
                                 index=['Mean', 'Avg.Std'])
print(SummaryHunger)

plt.figure(figsize=(22,4))
plt.plot(s2aff_group_low,label='valence')
plt.plot(s2exh_group_low,label='exhausted')
plt.plot(s2hun_group_low,label='hungry')
plt.legend()

plt.figure(figsize=(22,4))
plt.plot(s2aff_group_mid,label='valence')
plt.plot(s2exh_group_mid,label='exhausted')
plt.plot(s2hun_group_mid,label='hungry')
plt.legend()

plt.figure(figsize=(22,4))
plt.plot(s2aff_group_high,label='valence')
plt.plot(s2exh_group_high,label='exhausted')
plt.plot(s2hun_group_high,label='hungry')
plt.legend()



In [ ]:
from pandas.core.reshape.concat import concat
group_mid_cutoff = M - group_cutoff * 2
uo1T_group_detail_low = np.zeros((group_cutoff,4,N))
uo1T_group_detail_mid = np.zeros((group_mid_cutoff,4,N))
uo1T_group_detail_high = np.zeros((group_cutoff,4,N))

mean_uo1T_group_detail_low = np.zeros((group_cutoff,4))
mean_uo1T_group_detail_mid = np.zeros((group_mid_cutoff,4))
mean_uo1T_group_detail_high = np.zeros((group_cutoff,4))
mean_uo1T_group_detail = np.zeros((4,M))
print(uo1T_group_detail_low.shape)
for i in range(4):
  uo1T_group_detail_low[:,i,:] = uo1T_group[bool_low_gamma,i,:]
  mean_uo1T_group_detail_low[:,i] = uo1T_group[bool_low_gamma,i,:].mean(axis=1)
  uo1T_group_detail_mid[:,i,:] = uo1T_group[bool_mid_gamma,i,:]
  mean_uo1T_group_detail_mid[:,i] = uo1T_group[bool_mid_gamma,i,:].mean(axis=1)
  uo1T_group_detail_high[:,i,:] = uo1T_group[bool_high_gamma,i,:]
  mean_uo1T_group_detail_high[:,i] = uo1T_group[bool_high_gamma,i,:].mean(axis=1)
  mean_uo1T_group_detail[i,:] = np.concatenate((mean_uo1T_group_detail_low[:,i],
                                           mean_uo1T_group_detail_mid[:,i],
                                           mean_uo1T_group_detail_high[:,i]))  
uo1T_group_vec = np.concatenate((np.tile('low',group_cutoff),
                                np.tile('mid',(M - group_cutoff * 2)),
                                np.tile('high',group_cutoff)))
print(uo1T_group_vec.shape)
group_means = pd.DataFrame({'C1': mean_uo1T_group_detail[0],
                            'C2': mean_uo1T_group_detail[1],
                            'C3': mean_uo1T_group_detail[2],
                            'C4': mean_uo1T_group_detail[3],
                            'group': uo1T_group_vec})
print(group_means)

In [ ]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

mod1 = ols('C1 ~ group', data=group_means).fit()
mod2 = ols('C2 ~ group', data=group_means).fit()
mod3 = ols('C3 ~ group', data=group_means).fit()
mod4 = ols('C4 ~ group', data=group_means).fit()

aov_table1 = sm.stats.anova_lm(mod1, typ=2)
aov_table2 = sm.stats.anova_lm(mod2, typ=2)
aov_table3 = sm.stats.anova_lm(mod3, typ=2)
aov_table4 = sm.stats.anova_lm(mod4, typ=2)

#print(aov_table1,aov_table2,aov_table3,aov_table4)
#group_means.boxplot('C1', by='group', figsize=(12, 8))
#group_means.boxplot('C2', by='group', figsize=(12, 8))
#group_means.boxplot('C3', by='group', figsize=(12, 8))
#group_means.boxplot('C4', by='group', figsize=(12, 8))

In [ ]:
import pandas as pd

low_names = ['LowAgent_{}'.format(x) for x in range(group_cutoff)]
mid_names = ['MidAgent_{}'.format(x) for x in range(M - group_cutoff * 2)]
high_names = ['HighAgent_{}'.format(x) for x in range(group_cutoff)]
#b=pd.Panel(rollaxis(a,2)).to_frame()
#c=b.set_index(b.index.labels[0]).reset_index()
#c.columns=list('abc')
for i in range(4):
  globals()[f'Low_df_{i}'] = pd.DataFrame(data=uo1T_group_detail_low[:,i,:], 
                                          index=low_names,
                                          columns=range(2000))
  globals()[f'Low_df_{i}']['group']='low'
  globals()[f'Mid_df_{i}'] = pd.DataFrame(data=uo1T_group_detail_mid[:,i,:], 
                                          index=mid_names,
                                          columns=range(2000))
  globals()[f'Mid_df_{i}']['group']='mid'
  globals()[f'High_df_{i}'] = pd.DataFrame(data=uo1T_group_detail_high[:,i,:], 
                                          index=high_names,
                                          columns=range(2000))
  globals()[f'High_df_{i}']['group']='high'

  globals()[f'All_{i}'] = pd.concat([globals()[f'Low_df_{i}'], 
                                     globals()[f'Mid_df_{i}'], 
                                     globals()[f'High_df_{i}']], ignore_index=True, sort=False)

 #All_1 = pd.concat([Low_df_1, Mid_df_1, High_df_1], axis = 0)
print(All_0)

In [ ]:
#https://stackoverflow.com/questions/16592222/matplotlib-group-boxplots
#https://towardsdatascience.com/levenes-test-for-equality-of-variances-explained-with-python-examples-f0445a19805f

from pylab import plot, show, savefig, xlim, figure, hold, ylim, legend, boxplot, setp, axes

# function for setting the colors of the box plots pairs
def setBoxColors(bp):
    setp(bp['boxes'][0], color='blue')
    setp(bp['caps'][0], color='blue')
    setp(bp['caps'][1], color='blue')
    setp(bp['whiskers'][0], color='blue')
    setp(bp['whiskers'][1], color='blue')
    setp(bp['fliers'][0], color='blue')
    setp(bp['fliers'][1], color='blue')
    setp(bp['medians'][0], color='blue')

    setp(bp['boxes'][1], color='red')
    setp(bp['caps'][2], color='red')
    setp(bp['caps'][3], color='red')
    setp(bp['whiskers'][2], color='red')
    setp(bp['whiskers'][3], color='red')
    setp(bp['fliers'][2], color='red')
    setp(bp['fliers'][3], color='red')
    setp(bp['medians'][1], color='red')

# Some fake data to plot
A= [[1, 2, 5,],  [7, 2]]
B = [[5, 7, 2, 2, 5], [7, 2, 5]]
C = [[3,2,5,7], [6, 7, 3]]

fig = figure()
ax = axes()
hold(True)

# first boxplot pair
bp = boxplot(A, positions = [1, 2], widths = 0.6)
setBoxColors(bp)

# second boxplot pair
bp = boxplot(B, positions = [4, 5], widths = 0.6)
setBoxColors(bp)

# thrid boxplot pair
bp = boxplot(C, positions = [7, 8], widths = 0.6)
setBoxColors(bp)

# set axes limits and labels
xlim(0,9)
ylim(0,9)
ax.set_xticklabels(['A', 'B', 'C'])
ax.set_xticks([1.5, 4.5, 7.5])

# draw temporary red and blue lines and use them to create a legend
hB, = plot([1,1],'b-')
hR, = plot([1,1],'r-')
legend((hB, hR),('Apples', 'Oranges'))
hB.set_visible(False)
hR.set_visible(False)

savefig('boxcompare.png')
show()

#test

group_C0 = [[uo1T_group[bool_low_gamma,0,:]],[uo1T_group[bool_mid_gamma,0,:]],[uo1T_group[bool_high_gamma,0,:]]]
group_C1 = [[uo1T_group[bool_low_gamma,1,:]],[uo1T_group[bool_mid_gamma,1,:]],[uo1T_group[bool_high_gamma,1,:]]]
group_C2 = [[uo1T_group[bool_low_gamma,2,:]],[uo1T_group[bool_mid_gamma,2,:]],[uo1T_group[bool_high_gamma,2,:]]]
group_C3 = [[uo1T_group[bool_low_gamma,3,:]],[uo1T_group[bool_mid_gamma,3,:]],[uo1T_group[bool_high_gamma,3,:]]]

#group_low = [[uo1T_group[bool_low_gamma,0,:]],[uo1T_group[bool_low_gamma,1,:]],[uo1T_group[bool_low_gamma,2,:]],[uo1T_group[bool_low_gamma,3,:]]]
#group_mid = [[uo1T_group[bool_mid_gamma,0,:]],[uo1T_group[bool_mid_gamma,1,:]],[uo1T_group[bool_mid_gamma,2,:]],[uo1T_group[bool_mid_gamma,3,:]]]
#group_high = [[uo1T_group[bool_high_gamma,0,:]],[uo1T_group[bool_high_gamma,1,:]],[uo1T_group[bool_high_gamma,2,:]],[uo1T_group[bool_high_gamma,3,:]]]

group_low = [uo1T_group[bool_low_gamma,0,:],uo1T_group[bool_low_gamma,1,:],uo1T_group[bool_low_gamma,2,:],uo1T_group[bool_low_gamma,3,:]]
group_mid = [uo1T_group[bool_mid_gamma,0,:],uo1T_group[bool_mid_gamma,1,:],uo1T_group[bool_mid_gamma,2,:],uo1T_group[bool_mid_gamma,3,:]]
group_high = [uo1T_group[bool_high_gamma,0,:],uo1T_group[bool_high_gamma,1,:],uo1T_group[bool_high_gamma,2,:],uo1T_group[bool_high_gamma,3,:]]

data_groups = [group_C0, group_C1, group_C2, group_C3]
data_groups2 = [group_low,group_mid,group_high]
# --- Labels for your data:
ticks = ['C0','C1', 'C2','C3']
width       = 0.3
xlocations  = [ x*((1+ len(data_groups2))*width) for x in range(len(group_low)) ]

symbol      = 'r+'
ymin        = min ( [ val  for dg in data_groups2.all()  for data in dg for val in gamma_group.all() ] )
ymax        = max ( [ val  for dg in data_groups2  for data in dg for val in gamma_group ])

ax = pl.gca()
ax.set_ylim(ymin,ymax)

ax.grid(True, linestyle='dotted')
ax.set_axisbelow(True)

pl.xlabel('X axis label')
pl.ylabel('Y axis label')
pl.title('title')

space = len(data_groups)/2
offset = len(data_groups)/2


ax.set_xticks( xlocations )
ax.set_xticklabels( labels_list, rotation=0 )
# --- Offset the positions per group:

group_positions = []
for num, dg in enumerate(data_groups):    
    _off = (0 - space + (0.5+num))
    print(_off)
    group_positions.append([x-_off*(width+0.01) for x in xlocations])

for dg, pos in zip(data_groups, group_positions):
    pl.boxplot(dg, 
                sym=symbol,
    #            labels=['']*len(labels_list),
                labels=['']*len(labels_list),           
                positions=pos, 
                widths=width, 
    #           notch=False,  
    #           vert=True, 
    #           whis=1.5,
    #           bootstrap=None, 
    #           usermedians=None, 
    #           conf_intervals=None,
    #           patch_artist=False,
                )



pl.show()